In [ ]:
import pyautogui, collections
import pyaudio
from vosk import Model, KaldiRecognizer
import json
import time
import os
import winsound

Point = collections.namedtuple("Point", "x y")

############################# INPUTs #############################
skip_button_location = Point(x=1811, y=945)
skip_ad_wait_time = 0.5
MODEL_PATH = r"C:\Skip add\vosk-model-small-en-us-0.15"
##################################################################

def beep():
    winsound.Beep(1000, 200)  # frequency, duration

def skip_add_func():
    print(f"Skipping ad at: {skip_button_location}")
    pyautogui.click(skip_button_location)
    time.sleep(skip_ad_wait_time)

def get_input_device(pa):
    for i in range(pa.get_device_count()):
        info = pa.get_device_info_by_index(i)
        if info["maxInputChannels"] > 0:
            return i
    return None

def hotword():
    print("Checking Vosk model path...")

    if not os.path.exists(MODEL_PATH):
        print("❌ Model folder not found at:", MODEL_PATH)
        return

    print("Loading Vosk model...")
    model = Model(MODEL_PATH)
    recognizer = KaldiRecognizer(model, 16000)

    pa = pyaudio.PyAudio()
    device_index = get_input_device(pa)

    stream = pa.open(format=pyaudio.paInt16,
                     channels=1,
                     rate=16000,
                     input=True,
                     input_device_index=device_index,
                     frames_per_buffer=8000)

    stream.start_stream()

    print("Audio Recognition ON")

    while True:
        try:
            data = stream.read(4000, exception_on_overflow=False)
        except OSError:
            continue

        if recognizer.AcceptWaveform(data):
            result = json.loads(recognizer.Result())
            text = result.get("text", "").lower()

            if text.strip():
                print("You said:", text)

            if "jarvis" in text or "alexa" in text:
                print("Hotword detected! Skipping ad...")
                beep()
                skip_add_func()

hotword()


Checking Vosk model path...
Loading Vosk model...
Audio Recognition ON
You said: ah
You said: alexa
Hotword detected! Skipping ad...
Skipping ad at: Point(x=1811, y=945)
You said: you
